# 02 - CPT LoRA Adapter Training

This notebook trains Continual Pre-Training (CPT) LoRA adapters on: `cpt_corpus.jsonl`

Training is performed for multiple random seeds:
- 42
- 123
- 777

Only LoRA adapters are stored. The base model weights are not duplicated. Output structure: `artifacts/adapters/cpt/seed_<seed>/adapter/`.

In [ ]:
import exp_common as E

RUN_SEEDS = [777]  # then [42], [123]
MAX_SEQ_LENGTH = 2048
CPT_EPOCHS = 1.0
CPT_MAX_STEPS = -1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 1e-4

In [ ]:
rows = E.read_jsonl(E.PREPARED_DIR/'cpt_train.jsonl')
cpt_val = E.read_jsonl(E.PREPARED_DIR/'cpt_val.jsonl')

print('CPT rows:', len(rows))
print('CPT_val rows:', len(cpt_val))

for seed in RUN_SEEDS:
    out = E.ADAPTERS_DIR / 'cpt' / f'seed_{seed}'
    if (out/'adapter'/'adapter_model.safetensors').exists():
        print('Skip existing:', out/'adapter')
        continue
    print('Training CPT seed', seed)
    E.train_lora_adapter(
        rows, cpt_val, out, seed=seed, max_seq_length=MAX_SEQ_LENGTH,
        epochs=CPT_EPOCHS, max_steps=CPT_MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lora_r=16, lora_alpha=32, lora_dropout=0.05,
    )
print('Done')